# CLIPPR — synthetic PPR binder designer

**iGEM Marburg 2026.** Give it a target RNA sequence; it returns a PPR protein that binds
that sequence, a synthesisable coding sequence, a Golden Gate assembly plan, and the DNA
fragments to order.

Run the cells top to bottom. Cell 2 is the only one you normally edit.

---

**What the numbers mean, before you read any:**

- **Predicted fidelity** is computed from published ligation-count matrices
  (Pryor et al. 2020), *not* a measured assembly efficiency in your hands.
- **QC** is a sequence-complexity and feasibility check. It is not calibrated against
  vendor synthesis outcomes.
- **Cost** is an IDT oPools *list price*, not a quote. It is flat across pool sizes in
  this range, so it cannot be used to compare designs.
- Nothing in this tool has been validated at the bench.

## 1. Install

Guarded: if `clippr` already imports (a local editable install, or a re-run cell), this
does nothing. It never uses `--force-reinstall`, which would destroy a local install.

**If the repository is private**, add a GitHub token to Colab's **Secrets** (the key icon
in the left sidebar) under the name `GITHUB_TOKEN`, and enable it for this notebook.

Colab's permission to *open* a notebook from a private repo does **not** extend to the
runtime — the VM is a fresh machine with no GitHub credentials, so `pip` needs its own
token. Use a **fine-grained token, read-only, scoped to this one repository**. Never paste
a token into a cell; Secrets keeps it out of the notebook and out of version control.

The cell below works either way: with a secret set it installs from a private repo,
without one it falls back to the public URL. Nothing changes if the repo is made public
later.

In [ ]:
REPO = "SyedZainAliShah/clippr"

try:
    import clippr
    print(f"clippr {clippr.__version__} already available")
except ImportError:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # not on Colab, or no secret set — fall through to the public URL

    url = (f"git+https://{token}@github.com/{REPO}.git" if token
           else f"git+https://github.com/{REPO}.git")
    %pip install --quiet $url

    import clippr
    print(f"installed clippr {clippr.__version__}"
          f"{' (private repo, via Secrets)' if token else ' (public repo)'}")

## 2. Parameters

**`target_rna`** is the RNA sequence the PPR should bind. Its length sets the
architecture — one protein repeat per base, so 9 / 14 / 19 bases give 9S / 14S / 19S.

**`enzyme_profile`** decides which Type IIS recognition sites are excluded, and they are
different kinds of claim:

| profile | excludes | why |
|---|---|---|
| `assembly` | BsaI, BbsI | this assembly's own chemistry |
| `igem_rfc1000` *(default)* | + SapI | iGEM RFC[1000] requires BsaI and SapI absent |
| `moclo_compat` | + BsmBI | keeps later MoClo levels open — a preference, and it can
make some junctions infeasible |

In [ ]:
target_rna = "AAAAUGUGG"  #@param {type:"string"}
organism = "c_reinhardtii_nuclear"  #@param ["c_reinhardtii_nuclear"]
enzyme_profile = "igem_rfc1000"  #@param ["assembly", "igem_rfc1000", "moclo_compat"]
destination_level = "level0"  #@param ["level_minus1", "level0", "level1"]
seed = 42  #@param {type:"integer"}
write_files = True  #@param {type:"boolean"}

## 3. Design

One call. The pipeline picks cut positions and Golden Gate overhangs first, then
codon-optimises with those positions locked — that order matters, because optimising
first would let the optimiser rewrite the bases the junctions depend on.

In [ ]:
from clippr import DESTINATION_OVERHANGS, design_oneshot

result = design_oneshot(
    target_rna,
    organism=organism,
    enzyme_profile=enzyme_profile,
    destination=DESTINATION_OVERHANGS[destination_level],
    seed=seed,
    outdir="clippr_output" if write_files else None,
)
print(result["summary"])

## 4. Results

In [ ]:
qc = result["qc"]

print(f"target        {result['target_rna']}  ({result['architecture']})")
print(f"PPR code      {result['ppr_code']}")
print(f"protein       {len(result['protein'])} aa")
print(f"CDS           {len(result['cds'])} nt")
print(f"fragments     {len(result['oligos'])}")
print(f"cut positions {result['cuts']}")
print()
print(f"QC            {qc['status']}")
print(f"  GC          {qc['gc_pct']}%  (windows {qc['gc_window_min']}-{qc['gc_window_max']}%)")
print(f"  repeats     {qc['repeated_kmer_fraction']:.1%} of sequence in a repeated 20-mer")
print(f"  longest     {qc['longest_repeat']} nt exact repeat,"
      f" {qc['longest_homopolymer']} nt homopolymer")
print()
print(f"predicted fidelity  {result['fidelity']:.3f}   "
      f"(reaction: {' '.join(result['reaction_overhangs'])})")
print(f"list price          {result['cost']['total_eur']:.2f} EUR — not a quote")

for w in result["warnings"]:
    print(f"\n!  {w}")

In [ ]:
cols = ["fragment_id", "assembly_order", "aa_length", "oligo_length",
        "oh5_coding_site_5to3", "oh3_coding_site_5to3"]
result["oligos"][cols]

## 5. Why this design, and not another

The design carries its own decision record. For every junction it lists the overhangs it
could have used and what became of each:

- **selected** — the one used
- **considered** — feasible, but another scored at least as well
- **rejected** — *no* synonymous codon arrangement could avoid an excluded enzyme site, so
  the junction is impossible under the active profile, not merely worse

`local realizations` counts how many synonymous arrangements remain around a junction. It
is reported, never used to choose — but a junction with 2 is more fragile than one with 16,
and that is worth seeing before you order.

If a design looks surprising, read this rather than trusting it.

In [ ]:
audit = result["audit"]
print(audit.report())

rejected = audit.rejected
print(f"\n{len(rejected)} candidate overhang(s) ruled out entirely"
      f" under profile '{audit.enzyme_profile}'")
for d in rejected[:8]:
    print(f"    {d.sequence}  cut {d.junction_cut:>4d}   {d.reason}")
if len(rejected) > 8:
    print(f"    ... and {len(rejected) - 8} more")
if not rejected:
    print("    (every achievable overhang was usable at every junction)")

## 6. Download

Four files: the order CSV, the oligos as FASTA, the assembled gene, and an annotated
GenBank record you can open in Benchling or SnapGene.

In [ ]:
import os

if not result["paths"]:
    print("write_files was False — set it to True in cell 2 and re-run.")
else:
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None
    for kind, path in result["paths"].items():
        print(f"{kind:12s} {path}  ({os.path.getsize(path):,} bytes)")
        if colab_files:
            colab_files.download(path)

---

## Designing a whole library

For several targets at once. `crosstalk_report` shows how well separated the targets are
from one another — closely related targets risk one PPR binding another's UTR.

In [ ]:
targets = ["AAAAUGUGG", "GCUAAAGAC", "UUACACGUG"]  #@param

from clippr import crosstalk_report

print(crosstalk_report(targets))
print()
for t in targets:
    r = design_oneshot(t, organism=organism, seed=seed)
    print(f"{t}  {r['qc']['status']:8s} {len(r['oligos'])} fragments  "
          f"fidelity {r['fidelity']:.3f}")